# Retrieval-Augmented Generation (RAG) Project

This notebook implements a complete Retrieval-Augmented Generation (RAG) system
for document-based question answering.

Features:
- Document upload (TXT, PDF, DOCX) or text paste
- Chunking with overlap
- GPU-aware embedding generation
- Vector retrieval with FAISS
- Answer generation using OpenRouter LLM
- Hallucination prevention via prompt constraints
- Text-to-Speech output
- Interactive chat interface (Gradio)


## Why Retrieval-Augmented Generation (RAG)?

Large Language Models (LLMs) may generate incorrect or hallucinated answers.
RAG mitigates this problem by grounding the generation process in retrieved
document content.

This ensures:
- Factual correctness
- Transparency
- Reliability


In [1]:
!pip install -q sentence-transformers faiss-cpu gradio gtts openai PyPDF2 python-docx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 12.6 MB/s eta 0:00:00


In [2]:
import os
import faiss
import torch
import docx
import numpy as np
import gradio as gr
from gtts import gTTS
from openai import OpenAI
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer


## Step 1: OpenRouter API Setup


In [ ]:
os.environ["OPENROUTER_API_KEY"] = "your password"


In [4]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)


## Step 2: Document Loading


In [5]:
def read_txt_file(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def read_pdf_file(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text



def read_docx_file(path):
    doc = docx.Document(path)
    return "\n".join([p.text for p in doc.paragraphs])

In [6]:
def read_document(path):
    if path.endswith(".txt"):
        return read_txt_file(path)
    elif path.endswith(".pdf"):
        return read_pdf_file(path)
    elif path.endswith(".docx"):
        return read_docx_file(path)
    else:
        raise ValueError("Unsupported file format")


In [7]:
!gdown "https://drive.google.com/uc?id=1v4_2O15UPjaUEZO-7oijqSWFDnQiEG6t&confirm=t"

Downloading...
From: https://drive.google.com/uc?id=1v4_2O15UPjaUEZO-7oijqSWFDnQiEG6t&confirm=t
To: /content/yolov7_paper.pdf
100% 2.27M/2.27M [00:00<00:00, 17.6MB/s]


In [8]:
file_path = "/content/yolov7_paper.pdf"

text = read_document(file_path)

print("\n======Extracted PDF Content=======\n")
print(text[:500])


======Extracted PDF Content=======

YOLOv7: Trainable bag-of-freebies sets new state-of-the-art for real-time object
detectors
Chien-Yao Wang1, Alexey Bochkovskiy, and Hong-Yuan Mark Liao1
1Institute of Information Science, Academia Sinica, Taiwan
kinyiu@iis.sinica.edu.tw, alexeyab84@gmail.com, and liao@iis.sinica.edu.tw
Abstract
YOLOv7 surpasses all known object detectors in both
speed and accuracy in the range from 5 FPS to 160 FPS
and has the highest accuracy 56.8% AP among all known
real-time object detectors with 30 FPS or hi


## Step 3: Text Chunking with overlap


In [9]:
def chunk_text(text, chunk_size=500, overlap=100):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks


In [10]:
chunks = chunk_text(text, chunk_size=500, overlap=50)

print("Chunk-01", chunks[0])
print("Chunk-02", chunks[1])
print("Number of Chunks", len(chunks))

Chunk-01 YOLOv7: Trainable bag-of-freebies sets new state-of-the-art for real-time object
detectors
Chien-Yao Wang1, Alexey Bochkovskiy, and Hong-Yuan Mark Liao1
1Institute of Information Science, Academia Sinica, Taiwan
kinyiu@iis.sinica.edu.tw, alexeyab84@gmail.com, and liao@iis.sinica.edu.tw
Abstract
YOLOv7 surpasses all known object detectors in both
speed and accuracy in the range from 5 FPS to 160 FPS
and has the highest accuracy 56.8% AP among all known
real-time object detectors with 30 FPS or hi
Chunk-02 known
real-time object detectors with 30 FPS or higher on GPU
V100. YOLOv7-E6 object detector (56 FPS V100, 55.9%
AP) outperforms both transformer-based detector SWIN-
L Cascade-Mask R-CNN (9.2 FPS A100, 53.9% AP) by
509% in speed and 2% in accuracy, and convolutional-
based detector ConvNeXt-XL Cascade-Mask R-CNN (8.6
FPS A100, 55.2% AP) by 551% in speed and 0.7% AP
in accuracy, as well as YOLOv7 outperforms: YOLOR,
YOLOX, Scaled-YOLOv4, YOLOv5, DETR, Deformable
DETR, DINO-5s

## Step 4: Embedding Model (GPU-aware)


In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Step 5: Document Processing Pipeline


In [12]:
def process_document(file_path=None, text_input=""):
    global document_text, chunks, embeddings, index

    if file_path:
        try:
            document_text = read_document(file_path)
        except Exception as e:
            return f"❌ Error reading file: {str(e)}", gr.update(interactive=False)

    elif text_input.strip():
        document_text = text_input

    else:
        return "❌ Please upload a document or paste text.", gr.update(interactive=False)

    if not document_text.strip():
        return "❌ Document is empty.", gr.update(interactive=False)

    chunks = chunk_text(document_text)
    if len(chunks) == 0:
        return "❌ Chunking failed.", gr.update(interactive=False)

    embeddings = embedder.encode(
        chunks,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return f"✅ Document processed successfully ({len(chunks)} chunks).", gr.update(interactive=True)


## Step 6: Retrieval



In [13]:
def retrieve_chunks(query, k=3):
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]


## Step 7: Prompt Engineering (Hallucination Prevention)


In [14]:
def build_prompt(context, question):
    return f"""
You are a question-answering assistant.

Answer the question ONLY using the information provided in the context below.
If the answer is NOT in the context, say:
"I cannot find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
"""


## Step 8: Answer Generation (OpenRouter LLM)


In [19]:
def generate_answer(prompt):
    response = client.chat.completions.create(
       # model="mistralai/mistral-7b-instruct",
        model="openrouter/free",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    return response.choices[0].message.content


## Step 9: Text-to-Speech (TTS)


In [20]:
def text_to_speech(text, filename="answer.mp3"):
    tts = gTTS(text)
    tts.save(filename)
    return filename


## Step 10: Chat Pipeline



In [21]:
def chat(question):
    retrieved = retrieve_chunks(question)
    context = "\n\n".join(retrieved)
    prompt = build_prompt(context, question)
    answer = generate_answer(prompt)
    audio = text_to_speech(answer)
    return answer, audio


## Step 11: Chat Interface


In [22]:
with gr.Blocks() as demo:
    gr.Markdown("## RAG-based Document Question Answering System")

    file_input = gr.File(label="Upload Document (TXT, PDF, DOCX)", type="filepath")
    text_input = gr.Textbox(label="Or Paste Text", lines=6)

    process_btn = gr.Button("Process Document")
    status_box = gr.Textbox(label="Status")

    question_box = gr.Textbox(label="Ask a Question")
    answer_box = gr.Textbox(label="Answer")
    audio_box = gr.Audio(label="Answer Audio")

    process_btn.click(
        process_document,
        inputs=[file_input, text_input],
        outputs=[status_box, question_box]
    )

    question_box.submit(
        chat,
        inputs=question_box,
        outputs=[answer_box, audio_box]
    )

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b8bab2084aefad6005.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Conclusion

This notebook implements a complete, robust, and university-grade RAG system.
It supports multiple document formats, prevents hallucination, uses GPU-aware
embeddings, and provides both textual and audio responses through an interactive UI.
